# 스켈레톤 이미지 기반 CNN과 LSTM 필라테스 동작 인식 및 정확도 분석 모델


**CNN 모델 작성**

In [ ]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import GlobalAveragePooling2D
from tensorflow.keras.models import Model

base_cnn = MobileNetV2(weights='imagenet', include_top=False, input_shape=(224,224,3))
# CNN 출력 결과에 풀링 적용
x = GlobalAveragePooling2D()(base_cnn.output)
cnn_model = Model(inputs=base_cnn.input, outputs=x)
cnn_model.trainable = False


**이미지 마운트**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import cv2
import numpy as np
from sklearn.metrics import classification_report
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import GlobalAveragePooling2D
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.layers import LSTM, Dense, Masking
from tensorflow.keras.preprocessing.sequence import pad_sequences

base_cnn = MobileNetV2(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
x = GlobalAveragePooling2D()(base_cnn.output)
cnn_model = Model(inputs=base_cnn.input, outputs=x)
cnn_model.trainable = False

def load_frame_sequence(path, cnn_model):
    frames = sorted(os.listdir(path))
    features = []

    for f in frames:
        img_path = os.path.join(path, f)
        img = cv2.imread(img_path)

        if img is None:
            print(f"이미지 로드 실패: {img_path}")
            continue

        img = cv2.resize(img, (224, 224))
        img = img / 255.0
        features.append(img)

    if len(features) == 0:
        raise ValueError(f"유효한 이미지 없음: {path}")

    images = np.array(features)
    cnn_feats = cnn_model.predict(images, verbose=0)
    return cnn_feats


**데이터 처리**

In [ ]:
root_dir = '/content/drive/MyDrive/datasets/'

X, y = [], []
class_names = sorted(os.listdir(root_dir))
label_map = {idx: name for idx, name in enumerate(class_names)}

for idx, class_name in enumerate(class_names):
    label_map[idx] = class_name
    class_path = os.path.join(root_dir, class_name)

    for seq in os.listdir(class_path):
        seq_path = os.path.join(class_path, seq)
        if not os.path.isdir(seq_path):
            continue
        try:
            feat_seq = load_frame_sequence(seq_path, cnn_model)
            X.append(feat_seq)
            y.append(idx)
        except Exception as e:
            print(f"❌ 오류 발생: {seq_path}")
            print(e)


**LSTM 모델**

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Masking
X_padded = pad_sequences(X, padding='post', dtype='float32')
y = np.array(y)


model = Sequential([
    Masking(mask_value=0.0, input_shape=(X_padded.shape[1], X_padded.shape[2])),
    LSTM(64, return_sequences=True),
    LSTM(32),
    Dense(32, activation='relu'),
    Dense(len(label_map), activation='softmax')
])

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()


**학습**

In [ ]:


model.fit(X_padded, y, validation_split=0.2, epochs=15, batch_size=8)

**평가**

In [ ]:
from sklearn.metrics import classification_report

y_pred = model.predict(X_padded).argmax(axis=1)
print(classification_report(y, y_pred, target_names=list(label_map.values())))
